# egu_2026_talk.ipynb

Slide-friendly figures for the EGU 2026 talk.

Builds up Fig. 1 progressively (density → cluster → precipitation → organization),
then shows Fig. 5 with and without trajectories.

All figures saved to `paper/figures/talk/`.

## 1 · Imports

In [ ]:
import sys
sys.path.insert(0, '/Users/sophieabramian/Documents/troposwhere/paper')

import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize as Norm
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d

import workflow_paper as wfp
importlib.reload(wfp)

NUM_STATES    = wfp.NUM_STATES
state_colors  = wfp.state_colors
cluster_cmap  = wfp.cluster_cmap
norm_state    = wfp.norm_state

# ── Presentation-sized typography (applies to every figure in this notebook) ──
PRES_FS = 25
plt.rcParams.update({
    "font.size":        PRES_FS,
    "axes.titlesize":   PRES_FS,
    "axes.labelsize":   PRES_FS,
    "xtick.labelsize":  PRES_FS,
    "ytick.labelsize":  PRES_FS,
    "legend.fontsize":  PRES_FS,
    "legend.title_fontsize": PRES_FS,
    "figure.titlesize": PRES_FS,
    "xtick.major.size": 7, "xtick.major.width": 1.4,
    "ytick.major.size": 7, "ytick.major.width": 1.4,
})

# Talk-specific output directory
path_save_fig = os.path.join(wfp.FIGURES_DIR, 'talk')
os.makedirs(path_save_fig, exist_ok=True)

## 2 · Data loading

Same pipeline as `figures_paper.ipynb`: model inference, latents, diagnostics, transition matrix.

In [2]:
# ── Model & latents ─────────────────────────────────
loader, input_dim = wfp.load_dataloader()
model             = wfp.load_model(input_dim=input_dim)
out               = wfp.extract_latents(model, loader)
embeddings        = out["embeddings"]
states            = out["states"]
emb_pca, pca      = wfp.run_pca(embeddings)
_, state_keep_indices = wfp.keep_closest_latent_samples(embeddings, states)

# ── Diagnostics ─────────────────────────────────────
pw        = np.load(wfp.PATHS["pw"])[:len(states)]
prec      = np.load(wfp.PATHS["prec"])[:len(states)]
pw_norm   = Norm(vmin=1, vmax=17)
prec_norm = Norm(vmin=0, vmax=6)

# ── Transition analysis (used for Fig. 5 panels) ────────────────
Tmat = wfp.compute_transition_matrix(states, NUM_STATES, future=5)
pi   = wfp.stationary_distribution(Tmat)

## 3 · Shared layout helper

Each Fig. 1 variant uses the same axes layout (main scatter + marginal densities).
Defining it once keeps the slides visually consistent.

In [3]:
def make_fig1_axes(pca=None):
    """Build the Fig. 1 layout: main scatter + top/right marginal density panels.

    If `pca` is given, axis labels include the variance explained by each PC.
    """
    fig      = plt.figure(figsize=(9, 8))
    ax_main  = plt.axes([0.12, 0.12, 0.65, 0.65])
    ax_top   = plt.axes([0.12, 0.80, 0.65, 0.15], sharex=ax_main)
    ax_right = plt.axes([0.80, 0.12, 0.15, 0.65], sharey=ax_main)
    if pca is not None:
        evr = pca.explained_variance_ratio_
        ax_main.set_xlabel(f"PCA 1 ({evr[0]*100:.1f}% var)")
        ax_main.set_ylabel(f"PCA 2 ({evr[1]*100:.1f}% var)")
    else:
        ax_main.set_xlabel("PCA 1"); ax_main.set_ylabel("PCA 2")
    ax_top.set_ylabel("Density"); ax_right.set_xlabel("Density")
    ax_top.tick_params(axis="x", labelbottom=False)
    ax_right.tick_params(axis="y", labelleft=False)
    for a in (ax_main, ax_top, ax_right):
        a.grid(True, linestyle="--", alpha=0.35)
    return fig, ax_main, ax_top, ax_right


def plot_total_density(ax_top, ax_right, emb_pca, color="0.35", alpha_fill=0.25):
    """Plot one combined marginal density (all points, ignoring state)."""
    hist, bins = np.histogram(emb_pca[:, 0], bins=40, density=True)
    cx = (bins[:-1] + bins[1:]) / 2
    ax_top.fill_between(cx, gaussian_filter1d(hist, 1), alpha=alpha_fill, color=color)
    ax_top.plot(cx, gaussian_filter1d(hist, 1), color=color, lw=2)

    hist2, bins2 = np.histogram(emb_pca[:, 1], bins=40, density=True)
    cy = (bins2[:-1] + bins2[1:]) / 2
    ax_right.fill_betweenx(cy, gaussian_filter1d(hist2, 1), alpha=alpha_fill, color=color)
    ax_right.plot(gaussian_filter1d(hist2, 1), cy, color=color, lw=2)


def plot_state_densities(ax_top, ax_right, emb_pca, states):
    """Plot one marginal density curve per state, colored by cluster."""
    for s in range(NUM_STATES):
        pts = emb_pca[states == s]
        if len(pts) == 0:
            continue
        hist, bins = np.histogram(pts[:, 0], bins=40, density=True)
        cx = (bins[:-1] + bins[1:]) / 2
        ax_top.fill_between(cx, gaussian_filter1d(hist, 1), alpha=0.25, color=state_colors[s])
        ax_top.plot(cx, gaussian_filter1d(hist, 1), color=state_colors[s], lw=2)

        hist2, bins2 = np.histogram(pts[:, 1], bins=40, density=True)
        cy = (bins2[:-1] + bins2[1:]) / 2
        ax_right.fill_betweenx(cy, gaussian_filter1d(hist2, 1), alpha=0.25, color=state_colors[s])
        ax_right.plot(gaussian_filter1d(hist2, 1), cy, color=state_colors[s], lw=2)

## Slide 1 · Density only (`talk_fig1_density.pdf`)

Latent space with just the marginal densities and a neutral grey scatter — no clusters, no colors, no trajectories.

In [ ]:
from scipy.stats import gaussian_kde

fig, ax_main, ax_top, ax_right = make_fig1_axes(pca=pca)

# Color each point by the 2D KDE-estimated density (sorted so dense points draw on top)
xy      = emb_pca[:, :2].T
density = gaussian_kde(xy)(xy)
order   = np.argsort(density)
sc = ax_main.scatter(emb_pca[order, 0], emb_pca[order, 1], c=density[order],
                     cmap="viridis", s=28, alpha=0.85, edgecolors="none")

plot_total_density(ax_top, ax_right, emb_pca, color="0.35")

cbar_ax = fig.add_axes([0.12, -0.05, 0.65, 0.02])
cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
cbar.set_label("Point density")

plt.savefig(os.path.join(path_save_fig, "talk_fig1_density.pdf"), bbox_inches="tight")
plt.show()

## Slide 2 · Cluster colors, no trajectory (`talk_fig1_clusters.pdf`)

PCA scatter colored by HMM state, marginal densities per state, no trajectory overlay.

In [ ]:
fig, ax_main, ax_top, ax_right = make_fig1_axes(pca=pca)

unique, counts = np.unique(states, return_counts=True)
count_dict = dict(zip(unique.astype(int), counts.astype(int)))

for s in range(NUM_STATES):
    pts = emb_pca[states == s]
    ax_main.scatter(pts[:, 0], pts[:, 1], s=28, alpha=0.5, linewidth=0.25,
                    facecolors=state_colors[s],
                    label=f"State {s} (n={count_dict.get(s, 0)})")

plot_state_densities(ax_top, ax_right, emb_pca, states)

# Barycenter labels with the same repulsion as the paper figure
barycenters = {s: emb_pca[states == s].mean(axis=0)
               for s in range(NUM_STATES) if (states == s).any()}
shifted = np.array([barycenters[s] for s in range(NUM_STATES)])
for _ in range(200):
    for i in range(NUM_STATES):
        for j in range(i + 1, NUM_STATES):
            diff = shifted[i] - shifted[j]
            dist = np.linalg.norm(diff)
            if dist < 0.12:
                if dist < 1e-6:
                    diff = np.random.randn(2); dist = np.linalg.norm(diff)
                push = 0.003 * diff / dist
                shifted[i] += push; shifted[j] -= push
for s in range(NUM_STATES):
    ax_main.text(*shifted[s], str(s), fontsize=13, fontweight="bold",
                 color=state_colors[s], ha="center", va="center",
                 path_effects=[pe.withStroke(linewidth=2.6, foreground="white")])

sm_cb = ScalarMappable(cmap=cluster_cmap, norm=norm_state); sm_cb.set_array([])
cbar_ax = fig.add_axes([0.12, -0.05, 0.65, 0.02])
cbar = fig.colorbar(sm_cb, cax=cbar_ax, orientation='horizontal')
cbar.set_label("State Cluster")
cbar.set_ticks(np.arange(NUM_STATES))
cbar.set_ticklabels([str(i) for i in range(NUM_STATES)])

plt.savefig(os.path.join(path_save_fig, "talk_fig1_clusters.pdf"), bbox_inches="tight")
plt.show()

## Slide 3 · Colored by mean precipitation (`talk_fig1_prec.pdf`)

Same layout, but the main panel is colored by the mean precipitation diagnostic. Marginal densities show the total distribution.

In [ ]:
fig, ax_main, ax_top, ax_right = make_fig1_axes(pca=pca)

sc = ax_main.scatter(emb_pca[:, 0], emb_pca[:, 1], c=prec, cmap="Blues",
                     norm=prec_norm, s=28, alpha=0.7, edgecolors="none")
plot_total_density(ax_top, ax_right, emb_pca, color="0.35")

cbar_ax = fig.add_axes([0.12, -0.05, 0.65, 0.02])
cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
cbar.set_label("Mean precipitation")

plt.savefig(os.path.join(path_save_fig, "talk_fig1_prec.pdf"), bbox_inches="tight")
plt.show()

## Slide 4 · Colored by organization (`talk_fig1_organization.pdf`)

Main panel colored by precipitable-water variance (`var_PW`) — our proxy for convective organization.

In [ ]:
fig, ax_main, ax_top, ax_right = make_fig1_axes(pca=pca)

sc = ax_main.scatter(emb_pca[:, 0], emb_pca[:, 1], c=pw, cmap="rainbow",
                     norm=pw_norm, s=28, alpha=0.7, edgecolors="none")
plot_total_density(ax_top, ax_right, emb_pca, color="0.35")

cbar_ax = fig.add_axes([0.12, -0.05, 0.65, 0.02])
cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
cbar.set_label("Organization (PW variance)")

plt.savefig(os.path.join(path_save_fig, "talk_fig1_organization.pdf"), bbox_inches="tight")
plt.show()

## Slide 5 · Markov transitions, no trajectory (`talk_fig5_no_traj.pdf`)

Same as Fig. 5 of the paper (PCA scatter + Bezier transition arrows + transition matrix), but the example life-cycle trajectories are removed.

In [ ]:
FIGSIZE      = (18, 10)
BUBBLE_SCALE = 5000
TMIN, TMAX   = 0.01, 0.8

bubble_size = BUBBLE_SCALE * pi
evr = pca.explained_variance_ratio_

fig = plt.figure(figsize=FIGSIZE)
gs  = gridspec.GridSpec(100, 100, hspace=0.30, wspace=0.25)
ax_pca  = fig.add_subplot(gs[6:80,  4:46])
ax_mat  = fig.add_subplot(gs[6:80, 54:99])
cax_pca = fig.add_subplot(gs[92:94,  8:42])
cax_mat = fig.add_subplot(gs[92:94, 60:93])

sc = ax_pca.scatter(emb_pca[:, 0], emb_pca[:, 1], c=states,
                    cmap=cluster_cmap, norm=norm_state, s=28, alpha=0.55, linewidth=0.25)
ax_pca.set_xlabel(f"PCA 1 ({evr[0]*100:.1f}% var)")
ax_pca.set_ylabel(f"PCA 2 ({evr[1]*100:.1f}% var)")
ax_pca.grid(True, linestyle="--", alpha=0.35)

bary = np.array([emb_pca[states == s].mean(axis=0) if (states == s).any() else np.zeros(2)
                 for s in range(NUM_STATES)])

fig.colorbar(sc, cax=cax_pca, orientation="horizontal").set_label("Cluster state")

Tplot    = np.ma.masked_less(Tmat, TMIN)
norm_mat = mcolors.LogNorm(vmin=TMIN, vmax=TMAX)
im = ax_mat.imshow(Tplot, cmap="Blues", norm=norm_mat, origin="upper")
ax_mat.set(xlabel="Next state j", ylabel="Current state i")
for i in range(Tmat.shape[0]):
    for j in range(Tmat.shape[1]):
        ax_mat.text(j, i, f"{Tmat[i, j]:.2f}", ha="center", va="center", fontsize=18,
                    color="white" if Tmat[i, j] > 0.5 * Tmat.max() else "black")
fig.colorbar(im, cax=cax_mat, orientation="horizontal").set_label("P(i → j)")

wfp.draw_markov_transitions(ax_pca, Tmat, {s: bary[s] for s in range(NUM_STATES)},
                             min_weight=0.02, min_prob=0.011)

for s in range(NUM_STATES):
    ax_pca.scatter(*bary[s], s=bubble_size[s], color=state_colors[s],
                   edgecolors="black", linewidths=1.2, alpha=0.85, zorder=999)
    ax_pca.text(*bary[s], str(s), fontsize=16, fontweight="bold", color=state_colors[s],
                ha="center", va="center", zorder=1000,
                path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

for line in ax_pca.lines:
    line.set_rasterized(True)
plt.savefig(os.path.join(path_save_fig, "talk_fig5_no_traj.pdf"), bbox_inches="tight")
plt.show()

## Slide 6 · Markov transitions with trajectories (`talk_fig5_with_traj.pdf`)

Same as Slide 5, with the two example convective life-cycle trajectories overlaid (strongly vs. weakly organized).

In [ ]:
FIGSIZE         = (18, 10)
TRAJ_SMOOTH_WIN = 7
BUBBLE_SCALE    = 5000
TMIN, TMAX      = 0.01, 0.8

intervals    = [(10000, 10300), (5000, 5300)]
traj_colors  = ["k", "k"]
traj_styles  = ["--", ":"]

bubble_size = BUBBLE_SCALE * pi
evr = pca.explained_variance_ratio_

fig = plt.figure(figsize=FIGSIZE)
gs  = gridspec.GridSpec(100, 100, hspace=0.30, wspace=0.25)
ax_pca  = fig.add_subplot(gs[6:80,  4:46])
ax_mat  = fig.add_subplot(gs[6:80, 54:99])
cax_pca = fig.add_subplot(gs[92:94,  8:42])
cax_mat = fig.add_subplot(gs[92:94, 60:93])

sc = ax_pca.scatter(emb_pca[:, 0], emb_pca[:, 1], c=states,
                    cmap=cluster_cmap, norm=norm_state, s=28, alpha=0.55, linewidth=0.25)
ax_pca.set_xlabel(f"PCA 1 ({evr[0]*100:.1f}% var)")
ax_pca.set_ylabel(f"PCA 2 ({evr[1]*100:.1f}% var)")
ax_pca.grid(True, linestyle="--", alpha=0.35)

bary = np.array([emb_pca[states == s].mean(axis=0) if (states == s).any() else np.zeros(2)
                 for s in range(NUM_STATES)])

for ((t0, t1), col, style) in zip(intervals, traj_colors, traj_styles):
    traj = wfp.smooth_traj(emb_pca[t0:t1], win=TRAJ_SMOOTH_WIN)
    ax_pca.plot(traj[:, 0], traj[:, 1], color=col, linestyle=style, lw=2.4, zorder=20,
                path_effects=[pe.withStroke(linewidth=4.5, foreground="white")])

fig.colorbar(sc, cax=cax_pca, orientation="horizontal").set_label("Cluster state")

Tplot    = np.ma.masked_less(Tmat, TMIN)
norm_mat = mcolors.LogNorm(vmin=TMIN, vmax=TMAX)
im = ax_mat.imshow(Tplot, cmap="Blues", norm=norm_mat, origin="upper")
ax_mat.set(xlabel="Next state j", ylabel="Current state i")
for i in range(Tmat.shape[0]):
    for j in range(Tmat.shape[1]):
        ax_mat.text(j, i, f"{Tmat[i, j]:.2f}", ha="center", va="center", fontsize=18,
                    color="white" if Tmat[i, j] > 0.5 * Tmat.max() else "black")
fig.colorbar(im, cax=cax_mat, orientation="horizontal").set_label("P(i → j)")

wfp.draw_markov_transitions(ax_pca, Tmat, {s: bary[s] for s in range(NUM_STATES)},
                             min_weight=0.02, min_prob=0.011)

for s in range(NUM_STATES):
    ax_pca.scatter(*bary[s], s=bubble_size[s], color=state_colors[s],
                   edgecolors="black", linewidths=1.2, alpha=0.85, zorder=999)
    ax_pca.text(*bary[s], str(s), fontsize=16, fontweight="bold", color=state_colors[s],
                ha="center", va="center", zorder=1000,
                path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

traj_handles = [
    Line2D([0], [0], color="black", lw=2.4, linestyle="--", label="Strongly organized"),
    Line2D([0], [0], color="black", lw=2.4, linestyle=":",  label="Weakly organized"),
]
ax_pca.legend(handles=traj_handles, loc="lower left", frameon=True, framealpha=0.9, fontsize=14)

for line in ax_pca.lines:
    line.set_rasterized(True)
plt.savefig(os.path.join(path_save_fig, "talk_fig5_with_traj.pdf"), bbox_inches="tight")
plt.show()